In [1]:
import os
import requests
import logging
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

c:\Users\vedan\Desktop\GenAIProject\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class RepoState(TypedDict):
    owner: str
    repo: str
    files: List[str]
    current_file: str
    file_content: str
    chunks: List[Dict]
    issues: List[Dict]
    karma_score: float

In [3]:
def get_repo_files(state):
    logger.info("Getting repo files...")
    owner = state["owner"]
    repo = state["repo"]

    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/main?recursive=1"
    try:
        res = requests.get(url)
        res.raise_for_status()
        data = res.json()
    except Exception as e:
        logger.error(f"Failed to fetch repo files: {e}")
        return {"files": []}

    files = []
    for item in data.get("tree", []):
        if item["type"] == "blob" and item["path"].endswith((".py", ".js", ".ts", ".vue", ".java")):
            files.append(item["path"])

    logger.info(f"Found {len(files)} files")
    return {"files": files}

In [4]:
def select_file(state):
    if not state["files"]:
        logger.info("No files left to scan.")
        return {"current_file": None}
    file = state["files"].pop()
    logger.info(f"Selected file: {file}")
    return {"current_file": file}

In [6]:
def fetch_file_content(state):
    path = state["current_file"]
    if path is None:
        logger.warning("Current file is None, skipping fetch.")
        return {"file_content": ""}

    url = f"https://raw.githubusercontent.com/{state['owner']}/{state['repo']}/main/{path}"
    try:
        content = requests.get(url).text
        logger.info(f"Fetched {len(content.splitlines())} lines from {path}")
    except Exception as e:
        logger.error(f"Failed to fetch file content: {e}")
        content = ""
    return {"file_content": content}

In [7]:
def chunk_code(state):
    lines = state["file_content"].split("\n")
    chunks = []
    for i in range(0, len(lines), 40):
        chunk = "\n".join(lines[i:i+40])
        chunks.append({"code": chunk, "start_line": i+1})
    logger.info(f"Chunked file into {len(chunks)} chunks")
    return {"chunks": chunks}

In [8]:
logger.info("Loading TinyLlama model...")
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=-1,  # CPU
    max_length=1024,
    temperature=0.2,
    do_sample=False
)
logger.info("TinyLlama model loaded successfully!")

2026-03-08 22:45:49,362 [INFO] Loading TinyLlama model...
2026-03-08 22:45:50,490 [INFO] HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 22:45:50,568 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 468.38it/s, Materializing param=model.norm.weight]                              
2026-03-08 22:45:51,548 [INFO] HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 22:45:51,627 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
2026-03-08 22:45:51,982 [INFO] HTTP Reque

In [9]:
def scan_code(state):
    issues = state.get("issues", [])
    file = state["current_file"]
    logger.info(f"Scanning file: {file}")

    for i, chunk in enumerate(state["chunks"], start=1):
        logger.info(f"Scanning chunk {i}/{len(state['chunks'])} starting at line {chunk['start_line']}")
        prompt = f"""
You are a security code reviewer.
Find vulnerabilities in the following code.
Return JSON with:
issue, severity, line_number

Code:
{chunk['code']}
"""
        try:
            output = llm(prompt)[0]["generated_text"]
            if "issue" in output.lower():
                issues.append({
                    "file": file,
                    "line": chunk["start_line"],
                    "description": output
                })
        except Exception as e:
            logger.error(f"LLM failed on chunk starting line {chunk['start_line']}: {e}")

    logger.info(f"Found {len(issues)} issues so far")
    return {"issues": issues}

In [10]:
def compute_repo_score(state):
    score = 100
    for issue in state["issues"]:
        desc = issue["description"].lower()
        if "critical" in desc:
            score -= 10
        elif "medium" in desc:
            score -= 5
        else:
            score -= 2
    logger.info(f"Computed Karma Score: {max(score,0)}")
    return {"karma_score": max(score, 0)}

In [11]:
graph = StateGraph(RepoState)

graph.add_node("get_repo_files", get_repo_files)
graph.add_node("select_file", select_file)
graph.add_node("fetch_file_content", fetch_file_content)
graph.add_node("chunk_code", chunk_code)
graph.add_node("scan_code", scan_code)
graph.add_node("compute_repo_score", compute_repo_score)

graph.set_entry_point("get_repo_files")

graph.add_edge("get_repo_files","select_file")
graph.add_edge("select_file","fetch_file_content")
graph.add_edge("fetch_file_content","chunk_code")
graph.add_edge("chunk_code","scan_code")
graph.add_edge("scan_code","select_file")
graph.add_edge("select_file","compute_repo_score")

In [13]:
app = graph.compile()

result = app.invoke({
    "owner": "Vedant122003",
    "repo": "Screenshot-sorter-tool",
    "issues": []
})

logger.info(f"Repo Karma Score: {result['karma_score']}")
for issue in result["issues"]:
    logger.info(f"Issue found: {issue}")

2026-03-09 05:35:17,331 [INFO] Getting repo files...


2026-03-09 05:35:18,399 [INFO] Found 22 files
2026-03-09 05:35:18,401 [INFO] Selected file: sample1/views.py
2026-03-09 05:35:18,403 [INFO] Computed Karma Score: 100
2026-03-09 05:35:19,417 [INFO] Fetched 22 lines from sample1/views.py


KeyboardInterrupt: 